# 05 - Image and fusion thresholds: 90% and 95% recall

Run **06_fusion_model.ipynb first** to create fusion OOF predictions and checkpoints.
This notebook compares every CNN, CNN + metadata NN and CNN + metadata spline on
the identical development images. It reuses saved predictions and never retrains
models or reads test labels.

For each recall target, choose the threshold with maximum specificity subject
to development recall meeting the target. Also report threshold 0.5. Thresholds
are selected and measured on the same development scores: **tuning diagnostics,
not independent performance estimates**. Neither achieved recall nor a pooled
OOF threshold is guaranteed to transfer to a final model or fold ensemble.

In [1]:
import sys, json
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from melanoma_detection.evaluation import compare_validation_thresholds
from melanoma_detection.fusion_model import validate_fusion_predictions, predict_fusion_model
from melanoma_detection.splitting import load_splits

RESULTS = ROOT / "results"
RECALL_TARGETS = [0.90, 0.95]
REPORT_DIR = RESULTS / "thresholds"
MANIFEST_PATH = RESULTS / "metrics/fusion_all_models_manifest.json"
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError("Run 06_fusion_model.ipynb before threshold analysis.")
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
split_maps = [load_splits(path) for path in manifest["patient_fold_paths"]]

## Load and validate all matched OOF predictions

Image-only scores are taken from the same matched rows as fusion. Each checkpoint
is explicitly associated with its validation fold. Verify saved fusion fold
models reproduce their OOF predictions before tuning their thresholds.

In [2]:
tables, sources, checkpoints, labels = {}, {}, {}, {}
reference = None
for run, image_path in manifest["image_oof_paths"].items():
    variants = {}
    for variant in ("NN", "Spline"):
        path = RESULTS / "predictions/fusion" / run / f"fusion_{variant.lower()}_oof.csv"
        rows = pd.read_csv(path)
        validate_fusion_predictions(rows, rows, split_maps)
        if reference is None:
            reference = rows[["image_name", "patient_id", "target"]].copy()
        if set(rows.image_name) != set(reference.image_name):
            raise ValueError(f"{run}: different matched population.")
        rows = rows.set_index("image_name").loc[reference.image_name].reset_index()
        pd.testing.assert_frame_equal(rows[["image_name", "patient_id", "target"]], reference)
        variants[variant] = rows
        name = f"{run} | CNN + {variant}"
        tables[name] = rows.assign(score=rows.fusion_score, fold=rows.fusion_fold)
        sources[name] = str(path)
        labels[name] = (run, f"CNN + {variant}")
        for fold in (1, 2, 3):
            checkpoint = RESULTS / "models/fusion" / run / f"fusion_{variant.lower()}_fold_{fold}.joblib"
            model = joblib.load(checkpoint)
            held_out = rows.loc[rows.fusion_fold.eq(fold)]
            np.testing.assert_allclose(predict_fusion_model(model, held_out), held_out.fusion_score,
                                       rtol=1e-10, atol=1e-12)
            checkpoints[name, fold] = str(checkpoint)
    pd.testing.assert_frame_equal(variants["NN"][["image_name", "image_score", "fusion_fold"]],
                                  variants["Spline"][["image_name", "image_score", "fusion_fold"]])
    rows = variants["NN"]
    if "image_fold" not in rows or not rows.image_fold.eq(rows.fusion_fold).all():
        raise ValueError(f"{run}: image/fusion fold mappings differ; checkpoint thresholds need matching folds.")
    name = f"{run} | CNN only"
    tables[name] = rows.assign(score=rows.image_score, fold=rows.image_fold)
    sources[name] = image_path
    labels[name] = (run, "CNN only")
    for fold in (1, 2, 3):
        checkpoint = Path(image_path).parent / f"fold_{fold}" / run / "model.pt"
        if not checkpoint.is_file(): raise FileNotFoundError(checkpoint)
        checkpoints[name, fold] = str(checkpoint)
print(f"Validated {len(tables)} models on {len(reference)} common images.")

Validated 24 models on 6705 common images.


## Pooled development operating points

ROC-AUC and AP do not change with the threshold. For a shared recall target,
rank specificity to compare false-positive burden; retain CNN-only scores in
all reports. Pooled cutoffs are development candidates for the final inference
pipeline, not calibrated guarantees for a refitted stacker or fold-mean CNN.

In [3]:
pooled = compare_validation_thresholds(tables, reference.set_index("image_name").target, RECALL_TARGETS)
pooled["image_run"] = pooled.model.map(lambda name: labels[name][0])
pooled["variant"] = pooled.model.map(lambda name: labels[name][1])
pooled["scope"] = "pooled development OOF tuning; transfer to final pipeline unvalidated"
baselines = pooled.loc[pooled.variant.eq("CNN only"), ["image_run", "policy", "ROC_AUC", "AP", "specificity"]]
baselines = baselines.rename(columns={"ROC_AUC": "image_ROC_AUC", "AP": "image_AP", "specificity": "image_specificity"})
pooled = pooled.merge(baselines, on=["image_run", "policy"], validate="many_to_one")
pooled["specificity_change_vs_image"] = pooled.specificity - pooled.image_specificity
for target in RECALL_TARGETS:
    selected = pooled.loc[pooled.target_recall.eq(target)].sort_values("specificity", ascending=False)
    assert selected.recall.ge(target - 1e-12).all()
    print(f"Development recall target: {target:.0%}")
    display(selected[["image_run", "variant", "threshold", "recall", "specificity", "image_specificity",
                      "specificity_change_vs_image", "ROC_AUC", "image_ROC_AUC", "AP", "TN", "FP", "FN", "TP"]].round(4))

Development recall target: 90%
Development recall target: 95%


,image_run,variant,threshold,recall,specificity,image_specificity,specificity_change_vs_image,ROC_AUC,image_ROC_AUC,AP,TN,FP,FN,TP
16,densenet121_224_basic_hair0_b1f74d0a85,CNN only,0.0492,0.9055,0.5970,0.5970,0.0000,0.8127,0.8127,0.0753,3927,2651,12,115
13,densenet121_224_basic_hair0_b1f74d0a85,CNN + Spline,0.0121,0.9055,0.5762,0.5970,-0.0208,0.8237,0.8127,0.0877,3790,2788,12,115
61,resnet34_224_basic_hair0_3785453425,CNN only,0.0511,0.9055,0.5715,0.5715,0.0000,0.7744,0.7744,0.0477,3759,2819,12,115
49,resnet18_224_basic_hair0_012e640076,CNN + Spline,0.0125,0.9055,0.5711,0.5678,0.0033,0.8115,0.7899,0.0702,3757,2821,12,115
52,resnet18_224_basic_hair0_012e640076,CNN only,0.0246,0.9055,0.5678,0.5678,0.0000,0.7899,0.7899,0.0521,3735,2843,12,115
70,resnet50_224_basic_hair0_a035ef4bb7,CNN only,0.0303,0.9055,0.5634,0.5634,0.0000,0.8018,0.8018,0.0797,3706,2872,12,115
55,resnet34_224_basic_hair0_3785453425,CNN + NN,0.0116,0.9055,0.5515,0.5715,-0.0199,0.8011,0.7744,0.0708,3628,2950,12,115
7,convnext_tiny_224_basic_hair0_7790d73a11,CNN only,0.0527,0.9055,0.5439,0.5439,0.0000,0.8057,0.8057,0.0764,3578,3000,12,115
34,efficientnet_b3_300_basic_hair0_cffe5ac883,CNN only,0.0305,0.9055,0.5404,0.5404,0.0000,0.8263,0.8263,0.0890,3555,3023,12,115
46,resnet18_224_basic_hair0_012e640076,CNN + NN,0.0103,0.9055,0.5385,0.5678,-0.0293,0.8175,0.7899,0.0750,3542,3036,12,115


,image_run,variant,threshold,recall,specificity,image_specificity,specificity_change_vs_image,ROC_AUC,image_ROC_AUC,AP,TN,FP,FN,TP
53,resnet18_224_basic_hair0_012e640076,CNN only,0.0137,0.9528,0.5158,0.5158,0.0000,0.7899,0.7899,0.0521,3393,3185,6,121
17,densenet121_224_basic_hair0_b1f74d0a85,CNN only,0.0241,0.9528,0.4875,0.4875,0.0000,0.8127,0.8127,0.0753,3207,3371,6,121
8,convnext_tiny_224_basic_hair0_7790d73a11,CNN only,0.0390,0.9528,0.4628,0.4628,0.0000,0.8057,0.8057,0.0764,3044,3534,6,121
35,efficientnet_b3_300_basic_hair0_cffe5ac883,CNN only,0.0240,0.9528,0.4548,0.4548,0.0000,0.8263,0.8263,0.0890,2992,3586,6,121
62,resnet34_224_basic_hair0_3785453425,CNN only,0.0131,0.9528,0.4400,0.4400,0.0000,0.7744,0.7744,0.0477,2894,3684,6,121
50,resnet18_224_basic_hair0_012e640076,CNN + Spline,0.0057,0.9528,0.4337,0.5158,-0.0821,0.8115,0.7899,0.0702,2853,3725,6,121
47,resnet18_224_basic_hair0_012e640076,CNN + NN,0.0053,0.9528,0.4258,0.5158,-0.0900,0.8175,0.7899,0.0750,2801,3777,6,121
32,efficientnet_b3_300_basic_hair0_cffe5ac883,CNN + Spline,0.0068,0.9528,0.4255,0.4548,-0.0293,0.8254,0.8263,0.0955,2799,3779,6,121
71,resnet50_224_basic_hair0_a035ef4bb7,CNN only,0.0096,0.9528,0.4018,0.4018,0.0000,0.8018,0.8018,0.0797,2643,3935,6,121
11,densenet121_224_basic_hair0_b1f74d0a85,CNN + NN,0.0047,0.9528,0.3995,0.4875,-0.0880,0.8282,0.8127,0.0930,2628,3950,6,121


## Fold-specific thresholds and checkpoint bundles

Each fold cutoff is tuned on that checkpoint's validation scores, not on test
labels. Fusion remains non-nested because its inputs are existing base OOF scores.
The bundle contains the fold LR and fixed operating points; inference also needs
the correct image and metadata base models. Existing CNN weights are referenced,
not copied or modified. Final all-development stackers remain separate.

In [4]:
fold_reports = []
for name, table in tables.items():
    for fold, subset in table.groupby("fold"):
        report = compare_validation_thresholds({name: subset}, subset.set_index("image_name").target, RECALL_TARGETS)
        report["fold"] = int(fold)
        report["checkpoint"] = checkpoints[name, int(fold)]
        report["image_run"], report["variant"] = labels[name]
        report["source_predictions"] = sources[name]
        report["scope"] = "fold validation threshold tuning; not independent performance"
        fold_reports.append(report)
fold_thresholds = pd.concat(fold_reports, ignore_index=True)
assert fold_thresholds.loc[fold_thresholds.target_recall.notna(), "recall"].ge(
    fold_thresholds.loc[fold_thresholds.target_recall.notna(), "target_recall"] - 1e-12).all()
REPORT_DIR.mkdir(parents=True, exist_ok=True)
pooled.to_csv(REPORT_DIR / "pooled_threshold_comparison.csv", index=False)
fold_thresholds.to_csv(REPORT_DIR / "fold_thresholds.csv", index=False)
for (run, variant, fold), group in fold_thresholds.groupby(["image_run", "variant", "fold"]):
    if variant == "CNN only": continue
    source = Path(group.checkpoint.iloc[0])
    bundle = {"model": joblib.load(source), "thresholds": dict(zip(group.policy, group.threshold)),
              "source_checkpoint": str(source), "source_predictions": group.source_predictions.iloc[0],
              "fold": int(fold), "evaluation": "non-nested fold validation tuning",
              "recall_is_guaranteed_on_test": False}
    joblib.dump(bundle, source.with_name(source.stem + "_thresholds.joblib"))
(REPORT_DIR / "threshold_provenance.json").write_text(json.dumps({
    "fusion_manifest": str(MANIFEST_PATH), "sources": sources, "recall_targets": RECALL_TARGETS,
    "interpretation": "Development-only tuning. Pooled thresholds are unvalidated for final model/ensemble transfer."
}, indent=2), encoding="utf-8")
print("Saved pooled/fold thresholds and fold-fusion checkpoint bundles:", REPORT_DIR)
display(fold_thresholds.head(9).round(4))

Saved pooled/fold thresholds and fold-fusion checkpoint bundles: C:\Users\jjk91\OneDrive\Documents\GitHub\ADS2002-Melanoma\results\thresholds


,model,policy,threshold,target_recall,evaluation,AP,ROC_AUC,recall,specificity,precision,F1,TN,FP,FN,TP,average_recall,fold,checkpoint,image_run,variant,source_predictions,scope
0,convnext_tiny_224_basic_hair0_7790d73a11 | CNN + NN,fixed_0.5,0.5000,NaN,validation (threshold tuning),0.1451,0.8773,0.0000,1.0000,NaN,0.0000,2096,0,40,0,0.5000,1,C:\Users\jjk91\OneDrive\Documents\GitHub\ADS2002-Melanoma\results\models\fusion\convnext_tiny_224_basic_hair0_7790d73a11\fusion_nn_fold_1.joblib,convnext_tiny_224_basic_hair0_7790d73a11,CNN + NN,C:\Users\jjk91\OneDrive\Documents\GitHub\ADS2002-Melanoma\results\predictions\fusion\convnext_tiny_224_basic_hair0_7790d73a11\fusion_nn_oof.csv,fold validation threshold tuning; not independent performance
1,convnext_tiny_224_basic_hair0_7790d73a11 | CNN + NN,recall_0.9,0.0175,0.90,validation (threshold tuning),0.1451,0.8773,0.9000,0.6932,0.0530,0.1001,1453,643,4,36,0.7966,1,C:\Users\jjk91\OneDrive\Documents\GitHub\ADS2002-Melanoma\results\models\fusion\convnext_tiny_224_basic_hair0_7790d73a11\fusion_nn_fold_1.joblib,convnext_tiny_224_basic_hair0_7790d73a11,CNN + NN,C:\Users\jjk91\OneDrive\Documents\GitHub\ADS2002-Melanoma\results\predictions\fusion\convnext_tiny_224_basic_hair0_7790d73a11\fusion_nn_oof.csv,fold validation threshold tuning; not independent performance
2,convnext_tiny_224_basic_hair0_7790d73a11 | CNN + NN,recall_0.95,0.0137,0.95,validation (threshold tuning),0.1451,0.8773,0.9500,0.5782,0.0412,0.0790,1212,884,2,38,0.7641,1,C:\Users\jjk91\OneDrive\Documents\GitHub\ADS2002-Melanoma\results\models\fusion\convnext_tiny_224_basic_hair0_7790d73a11\fusion_nn_fold_1.joblib,convnext_tiny_224_basic_hair0_7790d73a11,CNN + NN,C:\Users\jjk91\OneDrive\Documents\GitHub\ADS2002-Melanoma\results\predictions\fusion\convnext_tiny_224_basic_hair0_7790d73a11\fusion_nn_oof.csv,fold validation threshold tuning; not independent performance
3,convnext_tiny_224_basic_hair0_7790d73a11 | CNN + NN,fixed_0.5,0.5000,NaN,validation (threshold tuning),0.0592,0.7833,0.0000,1.0000,NaN,0.0000,2183,0,44,0,0.5000,2,C:\Users\jjk91\OneDrive\Documents\GitHub\ADS2002-Melanoma\results\models\fusion\convnext_tiny_224_basic_hair0_7790d73a11\fusion_nn_fold_2.joblib,convnext_tiny_224_basic_hair0_7790d73a11,CNN + NN,C:\Users\jjk91\OneDrive\Documents\GitHub\ADS2002-Melanoma\results\predictions\fusion\convnext_tiny_224_basic_hair0_7790d73a11\fusion_nn_oof.csv,fold validation threshold tuning; not independent performance
4,convnext_tiny_224_basic_hair0_7790d73a11 | CNN + NN,recall_0.9,0.0134,0.90,validation (threshold tuning),0.0592,0.7833,0.9091,0.5071,0.0358,0.0690,1107,1076,4,40,0.7081,2,C:\Users\jjk91\OneDrive\Documents\GitHub\ADS2002-Melanoma\results\models\fusion\convnext_tiny_224_basic_hair0_7790d73a11\fusion_nn_fold_2.joblib,convnext_tiny_224_basic_hair0_7790d73a11,CNN + NN,C:\Users\jjk91\OneDrive\Documents\GitHub\ADS2002-Melanoma\results\predictions\fusion\convnext_tiny_224_basic_hair0_7790d73a11\fusion_nn_oof.csv,fold validation threshold tuning; not independent performance
5,convnext_tiny_224_basic_hair0_7790d73a11 | CNN + NN,recall_0.95,0.0098,0.95,validation (threshold tuning),0.0592,0.7833,0.9545,0.4517,0.0339,0.0655,986,1197,2,42,0.7031,2,C:\Users\jjk91\OneDrive\Documents\GitHub\ADS2002-Melanoma\results\models\fusion\convnext_tiny_224_basic_hair0_7790d73a11\fusion_nn_fold_2.joblib,convnext_tiny_224_basic_hair0_7790d73a11,CNN + NN,C:\Users\jjk91\OneDrive\Documents\GitHub\ADS2002-Melanoma\results\predictions\fusion\convnext_tiny_224_basic_hair0_7790d73a11\fusion_nn_oof.csv,fold validation threshold tuning; not independent performance
6,convnext_tiny_224_basic_hair0_7790d73a11 | CNN + NN,fixed_0.5,0.5000,NaN,validation (threshold tuning),0.0885,0.7815,0.0000,1.0000,NaN,0.0000,2299,0,43,0,0.5000,3,C:\Users\jjk91\OneDrive\Documents\GitHub\ADS2002-Melanoma\results\models\fusion\convnext_tiny_224_basic_hair0_7790d73a11\fusion_nn_fold_3.joblib,convnext_tiny_224_basic_hair0_7790d73a11,CNN + NN,C:\Users\jjk91\OneDrive\Documents\G

## Applying a saved fusion operating point

```python
bundle = joblib.load(".../fusion_nn_fold_1_thresholds.joblib")
probability = predict_fusion_model(bundle["model"], new_base_scores)
prediction = probability >= bundle["thresholds"]["recall_0.9"]
```

This applies a validation-selected cutoff; it does not promise 90% test recall.
Do not apply fold cutoffs to a final all-data stacker as though they calibrated it.
Final evaluation may report pooled development cutoffs as explicitly unvalidated
transfer policies, alongside 0.5. Never choose or adjust a threshold using test labels.